# Анализ жизненного цикла уязвимостей в экосистеме Python (PyPI)
## Разведочный анализ данных (Exploratory Data Analysis — EDA)

**Цель исследования:** изучение метрики **Time-to-Fix (TTF)** на основе сопоставления открытой базы уязвимостей OSV и реестра пакетов PyPI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Настройки графиков
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

### 1. Загрузка результирующего датасета `ttf_dataset.parquet`

In [ ]:
dataset_path = Path("../data/processed/ttf_dataset.parquet")
if not dataset_path.exists():
    dataset_path = Path("data/processed/ttf_dataset.parquet")

assert dataset_path.exists(), f"Файл {dataset_path} не найден! Выполните python src/match_data.py"
df = pd.read_parquet(dataset_path)
print(f"Датасет успешно загружен. Размер: {df.shape[0]} строк, {df.shape[1]} колонок.")

### 2. Структура данных и полнота атрибутов (Coverage)

In [ ]:
df.info()

In [ ]:
total_vulns = len(df)
with_fixed_version = df["matched_fixed_version"].notna().sum()
with_release_date = df["fixed_release_datetime"].notna().sum()
with_ttf = df["ttf_days"].notna().sum()

print("=== Анализ покрытия данных (Coverage) ===")
print(f"Всего записей об уязвимостях: {total_vulns}")
print(f"Найден фиксирующий релиз в PyPI: {with_fixed_version} ({with_fixed_version / total_vulns * 100:.2f}%)")
print(f"Успешно определена дата релиза: {with_release_date} ({with_release_date / total_vulns * 100:.2f}%)")
print(f"Рассчитана целевая метрика ttf_days: {with_ttf} ({with_ttf / total_vulns * 100:.2f}%)")

### 3. Распределение и квантили метрики Time-to-Fix (`ttf_days`)

In [ ]:
valid_ttf = df["ttf_days"].dropna()

print("=== Ключевые статистики метрики ttf_days ===")
print(f"Количество наблюдений: {len(valid_ttf)}")
print(f"Среднее (Mean):         {valid_ttf.mean():.2f} дней")
print(f"Медиана (Median):       {valid_ttf.median():.2f} дней")
print(f"Минимум (Min):          {valid_ttf.min():.2f} дней")
print(f"Максимум (Max):         {valid_ttf.max():.2f} дней")
print(f"10-й перцентиль (P10):  {valid_ttf.quantile(0.10):.2f} дней")
print(f"25-й перцентиль (Q1):   {valid_ttf.quantile(0.25):.2f} дней")
print(f"75-й перцентиль (Q3):   {valid_ttf.quantile(0.75):.2f} дней")
print(f"90-й перцентиль (P90):  {valid_ttf.quantile(0.90):.2f} дней")
print(f"95-й перцентиль (P95):  {valid_ttf.quantile(0.95):.2f} дней")

In [ ]:
# Анализ координированного раскрытия (патч вышел ДО публичной публикации в OSV)
cvd_count = (valid_ttf < 0).sum()
print(f"Случаев Coordinated Vulnerability Disclosure (ttf_days < 0): {cvd_count} ({cvd_count / len(valid_ttf) * 100:.2f}%)")

### 4. Топ-10 библиотек по количеству зарегистрированных уязвимостей

In [ ]:
top_packages = df["package_name"].value_counts().head(10)

print("=== Топ-10 библиотек по числу уязвимостей ===")
for rank, (pkg, count) in enumerate(top_packages.items(), 1):
    print(f"{rank:2d}. {pkg:<25} : {count} уязвимостей")

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=top_packages.values, y=top_packages.index, palette="viridis")
plt.title("Топ-10 пакетов PyPI по количеству уязвимостей в базе OSV", fontsize=13, weight="bold")
plt.xlabel("Количество уязвимостей")
plt.ylabel("Имя пакета")
plt.tight_layout()
plt.show()

### 5. Визуализация распределения TTF (для уязвимостей с ttf_days >= 0)

In [ ]:
positive_ttf = valid_ttf[valid_ttf >= 0]

plt.figure(figsize=(10, 5))
sns.histplot(positive_ttf[positive_ttf <= 365], bins=40, kde=True, color="royalblue")
plt.axvline(positive_ttf.median(), color="crimson", linestyle="--", label=f"Медиана ({positive_ttf.median():.1f} дн.)")
plt.title("Распределение Time-to-Fix (до 365 дней)", fontsize=13, weight="bold")
plt.xlabel("Дней до исправления (TTF)")
plt.ylabel("Количество уязвимостей")
plt.legend()
plt.tight_layout()
plt.show()